In [1]:
%pip install pandas
%pip install scikit-learn


[notice] A new release of pip is available: 23.2.1 -> 25.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 23.2.1 -> 25.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install dagshub
%pip install xgboost
# %pip install 


[notice] A new release of pip is available: 23.2.1 -> 25.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 23.2.1 -> 25.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier as RandomForest
from sklearn.metrics import classification_report
import pickle

data = '/Users/mj_peace/Desktop/MyMLOPS/Datasets/heart.csv'
df = pd.read_csv(data)
print(df.columns)

Index(['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach',
       'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target'],
      dtype='object')


In [4]:
df.shape
df.head()
X = df.drop('target',axis=1) # predictor feature coloumns
y = df.target


In [5]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10)
print('Training Set :',len(X_train))
print('Test Set :',len(X_test))
print('Training labels :',len(y_train))
print('Test Labels :',len(y_test))

Training Set : 242
Test Set : 61
Training labels : 242
Test Labels : 61


In [6]:
from sklearn.impute import KNNImputer
imputer = KNNImputer(n_neighbors=5)
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

In [7]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [8]:

from sklearn.ensemble import RandomForestClassifier as RandomForest

model = RandomForest(
    n_estimators=500,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=3,
    max_features="sqrt",
    class_weight="balanced",
    criterion="log_loss",
    random_state=10
)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test)

report = classification_report(y_test, y_pred)
print(report)

report_dict = classification_report(y_test, y_pred, output_dict=True)
print(report_dict)

              precision    recall  f1-score   support

           0       0.69      0.94      0.80        35
           1       0.85      0.42      0.56        26

    accuracy                           0.72        61
   macro avg       0.77      0.68      0.68        61
weighted avg       0.76      0.72      0.70        61

{'0': {'precision': 0.6875, 'recall': 0.9428571428571428, 'f1-score': 0.7951807228915663, 'support': 35.0}, '1': {'precision': 0.8461538461538461, 'recall': 0.4230769230769231, 'f1-score': 0.5641025641025641, 'support': 26.0}, 'accuracy': 0.7213114754098361, 'macro avg': {'precision': 0.7668269230769231, 'recall': 0.682967032967033, 'f1-score': 0.6796416434970651, 'support': 61.0}, 'weighted avg': {'precision': 0.7551229508196722, 'recall': 0.7213114754098361, 'f1-score': 0.696688392915926, 'support': 61.0}}


In [9]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")  
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.80
Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.80      0.82        35
           1       0.75      0.81      0.78        26

    accuracy                           0.80        61
   macro avg       0.80      0.80      0.80        61
weighted avg       0.81      0.80      0.80        61

Confusion Matrix:
[[28  7]
 [ 5 21]]


In [10]:
import joblib
model_filename = r"/Users/mj_peace/Desktop/MyMLOPS/Datasets/heart_model.pkl"  # Ensure the path is correct and does not have extra quotes'
joblib.dump(model, model_filename)
print(f"Model saved as {model_filename}")


Model saved as /Users/mj_peace/Desktop/MyMLOPS/Datasets/heart_model.pkl


In [11]:
%pip install mlflow



[notice] A new release of pip is available: 23.2.1 -> 25.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [12]:
import mlflow
import mlflow.sklearn
# Set the tracking URI to the local file system
mlflow.set_tracking_uri("file:./mlruns")
# Set the experiment name
mlflow.set_experiment("2408_heart_disease_experiment")
# Start an MLflow run
with mlflow.start_run():
    # Log the model
    mlflow.sklearn.log_model(model, "rf_diabetes_model")
    # Log parameters
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("random_state", 42)
    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    # Log the model file
    mlflow.log_artifact(model_filename, artifact_path="model_files")
# Print the run ID
    run_id = mlflow.active_run().info.run_id
print(f"Run ID: {run_id}")

2025/08/23 22:17:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/08/23 22:17:31 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Run ID: 26382f2966074f1b92ee1389e28e655f
